In [ ]:
import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import (
    BitsAndBytesConfig,
    Gemma4ForConditionalGeneration,
    Gemma4Processor
)
from trl.trainer.sft_config import SFTConfig
from trl.trainer.sft_trainer import SFTTrainer

In [ ]:
dataset = load_dataset("json", data_files="dataset.jsonl", split="train")

dataset = dataset.shuffle(seed=42)
split = dataset.train_test_split(test_size=0.1, seed=42)

In [ ]:
model_id = "google/gemma-4-E2B-it"

processor = Gemma4Processor.from_pretrained(model_id)

model = Gemma4ForConditionalGeneration.from_pretrained(
    model_id,
    dtype="auto",
    device_map="auto",
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_storage=torch.bfloat16,
    ),
)

In [ ]:
args = SFTConfig(
    output_dir="gemma-4-E2B-it-finetune",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=1,
    eval_accumulation_steps=4,
    gradient_checkpointing=True,
    bf16=True,
    # tf32=True,
    max_length=512,
    lr_scheduler_type="constant",
    learning_rate=5e-5,
    max_grad_norm=0.3,
    eval_strategy="steps",
    eval_steps=0.1,
    save_steps=0.1,
    save_total_limit=1,
    logging_steps=5,
    # report_to="trackio",
)

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=16,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
    # make sure to save the lm_head and embed_tokens as you train the special tokens
    modules_to_save=["lm_head", "embed_tokens"],
    ensure_weight_tying=True,
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    peft_config=peft_config,
    processing_class=processor,
)

In [ ]:
trainer.train()